# Immigration to Canada (UN Data, 1980-2013)

**Goal:** Read the UN immigration data for Canada, clean it, and reshape it so each row is a country with its continent, region, development name, yearly immigrant counts (1980-2013) and a Total column.

In [1]:
# pandas is used to read and reshape the data
import pandas as pd

# Google Sheet exported as an Excel file
file_url = 'https://docs.google.com/spreadsheets/d/'\
           '1OGs8KoHGEZDxyFdaxAKmZbeO1BiSWVQ6/'\
           'export?format=xlsx'

# Read the 'Canada by Citizenship' sheet.
# skiprows=20 skips the 20 title/logo rows, so the column
# names on row 21 become the header of the DataFrame
df = pd.read_excel(file_url,
                   sheet_name='Canada by Citizenship',
                   skiprows=20)

In [2]:
# Size of the table (rows, columns)
print("Shape:", df.shape)

# First five rows
print(df.head())

# Last five rows (checks for footnote rows at the bottom)
print(df.tail())

# All column names
print(list(df.columns))

Shape: (197, 43)
         Type    Coverage          OdName  AREA AreaName   REG  \
0  Immigrants  Foreigners     Afghanistan   935     Asia  5501   
1  Immigrants  Foreigners         Albania   908   Europe   925   
2  Immigrants  Foreigners         Algeria   903   Africa   912   
3  Immigrants  Foreigners  American Samoa   909  Oceania   957   
4  Immigrants  Foreigners         Andorra   908   Europe   925   

           RegName  DEV             DevName  1980  ...  2004  2005  2006  \
0    Southern Asia  902  Developing regions    16  ...  2978  3436  3009   
1  Southern Europe  901   Developed regions     1  ...  1450  1223   856   
2  Northern Africa  902  Developing regions    80  ...  3616  3626  4807   
3        Polynesia  902  Developing regions     0  ...     0     0     1   
4  Southern Europe  901   Developed regions     0  ...     0     0     1   

   2007  2008  2009  2010  2011  2012  2013  
0  2652  2111  1746  1758  2203  2635  2004  
1   702   560   716   561   539   620

## Step 1: Inspect the data types and find the non-country rows
Before cleaning, we check what type of data each column holds, whether any values are missing, and which rows are not real countries.

In [3]:
# How many columns hold text (object) vs whole numbers (int64)
print(df.dtypes.value_counts())

# Count of empty (missing) values in the whole table
print("\nMissing values:", df.isnull().sum().sum())

# Rows that are NOT single countries (Coverage is not 'Foreigners')
print("\nNon-country rows:")
print(df[df['Coverage'] != 'Foreigners'][['OdName', 'Coverage']])

int64    37
str       6
Name: count, dtype: int64

Missing values: 0

Non-country rows:
     OdName  Coverage
32   Canada  Citizens
196   Total      Both


## Step 2: Clean the table
1. Save the UN's own 'Total' row (used later to double-check our math), then remove it from the table because it is a sum of all countries, not a country.
2. Drop the columns that are not in the target table: Type, Coverage, AREA, REG and DEV.
3. Rename OdName to Country, AreaName to Continent and RegName to Region.
4. Convert the year column names from numbers to text and use Country as the row label (index).

In [4]:
# Save the UN 'Total' row so we can verify our results later
un_total_row = df[df['OdName'] == 'Total']

# Remove the 'Total' row (an aggregate, not a country)
df = df[df['OdName'] != 'Total'].copy()

# Drop the columns that are not needed in the final table
df = df.drop(columns=['Type', 'Coverage', 'AREA', 'REG', 'DEV'])

# Rename columns to the names used in the target table
df = df.rename(columns={'OdName': 'Country',
                        'AreaName': 'Continent',
                        'RegName': 'Region'})

# Make every column name text, so '1980' (text) is used
# instead of 1980 (number) when we select year columns
df.columns = df.columns.map(str)

# Use Country as the row label (index)
df = df.set_index('Country')

print("Shape after cleaning:", df.shape)
print(list(df.columns[:5]))

Shape after cleaning: (196, 37)
['Continent', 'Region', 'DevName', '1980', '1981']


## Step 3: Add the Total column
The Total column is the sum of immigrants from 1980 to 2013 for each country. We build a list of the year columns and sum across each row (axis=1).

In [5]:
# List of year columns as text: '1980', '1981', ..., '2013'
years = [str(year) for year in range(1980, 2014)]

# Sum each row across all year columns
df['Total'] = df[years].sum(axis=1)

# Show the first five rows of the final table
print(df.head())
print("\nFinal shape:", df.shape)

               Continent           Region             DevName  1980  1981  \
Country                                                                     
Afghanistan         Asia    Southern Asia  Developing regions    16    39   
Albania           Europe  Southern Europe   Developed regions     1     0   
Algeria           Africa  Northern Africa  Developing regions    80    67   
American Samoa   Oceania        Polynesia  Developing regions     0     1   
Andorra           Europe  Southern Europe   Developed regions     0     0   

                1982  1983  1984  1985  1986  ...  2005  2006  2007  2008  \
Country                                       ...                           
Afghanistan       39    47    71   340   496  ...  3436  3009  2652  2111   
Albania            0     0     0     0     1  ...  1223   856   702   560   
Algeria           71    69    63    44    69  ...  3626  4807  3623  4005   
American Samoa     0     0     0     0     0  ...     0     1     0     0  

## Step 4: Verify the result
We compare our Total values with the target table in the assignment (Afghanistan 58639, Albania 15699, Algeria 69439, American Samoa 6, Andorra 15). We also compare the sum of all country totals with the UN's own 'Total' row.

In [6]:
# Totals for the first five countries
print(df.loc[['Afghanistan', 'Albania', 'Algeria',
              'American Samoa', 'Andorra'], 'Total'])

# Sum of all country totals vs the UN's own total row
# (columns from position 9 onward are the year columns)
print("\nSum of all countries:", df['Total'].sum())
print("UN Total row sum:   ",
      un_total_row.iloc[:, 9:].sum(axis=1).values[0])

Country
Afghanistan       58639
Albania           15699
Algeria           69439
American Samoa        6
Andorra              15
Name: Total, dtype: int64

Sum of all countries: 6924212
UN Total row sum:    6924212


## Conclusion
The raw UN spreadsheet was converted into the requested table: one row per country, with its Continent, Region, DevName, yearly immigrant counts from 1980 to 2013, and a Total column.

To do this we skipped the 20 title rows, removed the aggregate 'Total' row, dropped the unneeded columns (Type, Coverage, AREA, REG, DEV), renamed OdName, AreaName and RegName to Country, Continent and Region, used Country as the index, and added a Total column by summing the year columns.

**Verification:** The Total values for Afghanistan (58639), Albania (15699), Algeria (69439), American Samoa (6) and Andorra (15) match the target table in the assignment. The sum of all country totals (6,924,212) also equals the UN's own 'Total' row, so no data was lost during the conversion.